In [11]:
from SymbolicDSGE import DSGESolver, ModelParser
from SymbolicDSGE.estimation.spec import (
    EstimationParameterSpec,
    EstimationSpec,
    OptimizationResultMeta,
    PriorSpec,
)
from SymbolicDSGE.monte_carlo.spec import EdgeSpec, NodeSpec, PipelineSpec
from SymbolicDSGE.bundle import ShockGeneration, SimSpec
from SymbolicDSGE import BundleBuilder
import numpy as np
from numpy import array, float64

import io
import pandas as pd

In [ ]:
parser = ModelParser("../../MODELS/POST82.yaml")
model, kalman = parser.get_all()

solver = DSGESolver(model, kalman)
compiled = solver.compile(
    linearize=False,
)
sol = solver.solve(
    compiled,
    steady_state=array([0.0, 0.0, 0.0, 0.0, 0.0], dtype=float64),
)

In [ ]:
estimation_spec = EstimationSpec(
    method="map",
    parameters=[
        EstimationParameterSpec(
            name="psi_pi",
            initial=1.5,
            estimate=True,
            lower=1.0,
            upper=3.0,
            prior=PriorSpec(
                distribution="normal",
                parameters={"loc": 1.5, "scale": 0.25},
            ),
        ),
        EstimationParameterSpec(
            name="psi_x",
            initial=0.5,
            estimate=True,
            lower=0.0,
            upper=1.0,
            prior=PriorSpec(
                distribution="normal",
                parameters={"loc": 0.5, "scale": 0.2},
            ),
        ),
    ],
    observables=["Infl", "Rate"],
    method_kwargs={"options": {"maxiter": 50}},
)

rng = np.random.default_rng(0)
observed = rng.standard_normal((40, 2))
estimation_result_meta = OptimizationResultMeta(
    kind="map",
    theta={"psi_pi": 1.48, "psi_x": 0.55},
    success=True,
    message="Optimization converged.",
    fun=-87.3,
    loglik=-85.1,
    logprior=-2.2,
    logpost=-87.3,
    nfev=124,
    nit=14,
)

In [ ]:
mc_pipeline = PipelineSpec(
    nodes=[
        NodeSpec(
            id="sim",
            step_type="simulation",
            name="DGP Simulation",
            params={"T": 100},
        ),
        NodeSpec(
            id="jb",
            step_type="jarque_bera",
            name="Normality Check",
            params={"source": "observables"},
        ),
    ],
    edges=[EdgeSpec(source="sim", target="jb")],
)

In [ ]:
simulation = SimSpec(
    role="reference",
    T=25,
    observables=True,
    shock_scale=1.0,
    shock_generation=ShockGeneration(
        dist="norm",
        seed=42,
        loc=0.0,
    ),
)

In [10]:
bundle_path = (
    BundleBuilder(created_by="experiment-1")  # (1)!
    .add_model(
        "reference",
        model.source_yaml,  # (2)!
        compile_kwargs={"linearize": False},
    )
    .add_estimation(
        estimation_spec,
        result=estimation_result_meta,
        observed=observed,
        observable_names=["Infl", "Rate"],  # (3)!
    )
    .add_mc(mc_pipeline)
    .set_simulation(simulation)
    .write("experiment-1.sdsge")
)

print("Bundle written to:", bundle_path)

Bundle written to: experiment-1.sdsge


In [13]:
aux = pd.DataFrame(
    {
        "date": pd.date_range("2000-01-01", periods=40, freq="QS"),
        "gdp_growth": rng.standard_normal(40),
    }
)
csv_buffer = io.StringIO()
aux.to_csv(csv_buffer, index=False)

BundleBuilder().add_model("reference", model.source_yaml).add_raw_data(
    "auxiliary_series", csv_buffer.getvalue()
).write("with-raw-data.sdsge")

WindowsPath('with-raw-data.sdsge')

In [14]:
!unzip -l experiment-1.sdsge

Archive:  experiment-1.sdsge
  Length      Date    Time    Name
---------  ---------- -----   ----
     1627  13-06-2026 12:29   manifest.json
     2599  13-06-2026 12:29   model/reference.yaml
      864  13-06-2026 12:29   estimation/spec.json
      305  13-06-2026 12:29   estimation/result.json
     1380  13-06-2026 12:29   estimation/observed.parquet
      389  13-06-2026 12:29   montecarlo/pipeline.json
---------                     -------
     7164                     6 files
